### Data Cleaning

Before starting the analysis, I first reviewed the datasets to make sure the data quality was good enough to work with. This step includes checking the dataset structure, confirming the column names loaded correctly, and looking for missing values or inconsistencies that could affect the analysis.

In [2]:
import pandas as pd

In [5]:
purchases = pd.read_csv("../data/purchases.csv")
purchase_prices = pd.read_csv("../data/purchase_prices.csv")
vendor_summary = pd.read_csv("../data/vendor_sales_summary.csv")

In [12]:
sales_sample = pd.read_csv("../data/sales.csv", nrows=100000)

In [13]:
print("purchases shape:", purchases.shape)
print("purchase_prices shape:", purchase_prices.shape)
print("vendor_summary shape:", vendor_summary.shape)
print("sales_sample shape:", sales_sample.shape)

purchases shape: (2372474, 16)
purchase_prices shape: (12261, 9)
vendor_summary shape: (10692, 18)
sales_sample shape: (100000, 14)


In [13]:
purchases.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber',
       'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate',
       'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification'],
      dtype='str')

In [16]:
purchase_prices.columns

Index(['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification',
       'PurchasePrice', 'VendorNumber', 'VendorName'],
      dtype='str')

In [15]:
vendor_summary.columns

Index(['VendorNumber', 'VendorName', 'Brand', 'Description', 'PurchasePrice',
       'ActualPrice', 'Volume', 'TotalPurchaseQuantity',
       'TotalPurchaseDollars', 'TotalSalesQuantity', 'TotalSalesPrice',
       'TotalSalesDollars', 'TotalExciseTax', 'FreightCost', 'GrossProfit',
       'ProfitMargin', 'StockTurnover', 'SalestoPurchaseRatio'],
      dtype='str')

In [14]:
sales_sample.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity',
       'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification',
       'ExciseTax', 'VendorNo', 'VendorName'],
      dtype='str')

In [17]:
purchases.isnull().sum()

InventoryId       0
Store             0
Brand             0
Description       0
Size              3
VendorNumber      0
VendorName        0
PONumber          0
PODate            0
ReceivingDate     0
InvoiceDate       0
PayDate           0
PurchasePrice     0
Quantity          0
Dollars           0
Classification    0
dtype: int64

In [18]:
purchase_prices.isnull().sum()

Brand             0
Description       1
Price             0
Size              1
Volume            1
Classification    0
PurchasePrice     0
VendorNumber      0
VendorName        0
dtype: int64

In [19]:
vendor_summary.isnull().sum()

VendorNumber             0
VendorName               0
Brand                    0
Description              0
PurchasePrice            0
ActualPrice              0
Volume                   0
TotalPurchaseQuantity    0
TotalPurchaseDollars     0
TotalSalesQuantity       0
TotalSalesPrice          0
TotalSalesDollars        0
TotalExciseTax           0
FreightCost              0
GrossProfit              0
ProfitMargin             0
StockTurnover            0
SalestoPurchaseRatio     0
dtype: int64

In [20]:
sales_sample.isnull().sum()

InventoryId       0
Store             0
Brand             0
Description       0
Size              0
SalesQuantity     0
SalesDollars      0
SalesPrice        0
SalesDate         0
Volume            0
Classification    0
ExciseTax         0
VendorNo          0
VendorName        0
dtype: int64

### Data Quality Summary

After inspecting the datasets, the data appears to be largely clean and well structured. Only a very small number of missing values were found in fields like product size and description, and these represent a tiny portion of the overall data.

Because the missing values are minimal and do not affect the key fields used in the analysis, no major cleaning steps were required. The datasets are therefore suitable for further analysis.

In [25]:
validation_check = purchases["PurchasePrice"] * purchases["Quantity"]
difference = (validation_check - purchases["Dollars"]).abs()

difference.describe()

count    2.372474e+06
mean     5.545287e-15
std      2.448177e-14
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.637979e-12
dtype: float64

### Purchase Value Validation

I checked whether the `Dollars` column correctly represents the total purchase value for each transaction.  
In theory, the total purchase value should equal:

PurchasePrice × Quantity

To verify this, I calculated the expected value using the price and quantity columns and compared it with the recorded `Dollars` column. The differences were essentially zero, with only extremely small rounding differences due to floating-point calculations.

This confirms that the purchase totals in the dataset are internally consistent and can be used reliably in the analysis.

In [26]:
sales_validation = sales_sample["SalesPrice"] * sales_sample["SalesQuantity"]
sales_difference = (sales_validation - sales_sample["SalesDollars"]).abs()

sales_difference.describe()

count    1.000000e+05
mean     3.297629e-16
std      5.647127e-15
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      9.094947e-13
dtype: float64

### Sales Revenue Validation

I also verified that the `SalesDollars` column correctly represents the total revenue for each transaction.

For each row, the total sales value should equal: SalesPrice × SalesQuantity

After calculating the expected revenue and comparing it with the recorded `SalesDollars` values, the differences were essentially zero. The very small differences that appear are due to floating-point rounding and have no meaning.

This confirms that the sales revenue values are internally consistent and can be used reliably in the analysis.

In [30]:
vendor_check = purchases.groupby("VendorNumber")["VendorName"].nunique()
vendor_check[vendor_check > 1]

VendorNumber
1587    2
2000    2
4425    2
Name: VendorName, dtype: int64

In [35]:
purchases[purchases["VendorNumber"] == 1587]["VendorName"].unique()

<StringArray>
['VINEYARD BRANDS INC        ', 'VINEYARD BRANDS LLC        ']
Length: 2, dtype: str

In [37]:
purchases[purchases["VendorNumber"] == 2000]["VendorName"].unique()

<StringArray>
['SOUTHERN WINE & SPIRITS NE ', 'SOUTHERN GLAZERS W&S OF NE ']
Length: 2, dtype: str

In [38]:
purchases[purchases["VendorNumber"] == 4425]["VendorName"].unique()

<StringArray>
['MARTIGNETTI COMPANIES ', 'MARTIGNETTI COMPANIES']
Length: 2, dtype: str

In [42]:
purchases["VendorName"] = purchases["VendorName"].str.strip()
sales_sample["VendorName"] = sales_sample["VendorName"].str.strip()
purchase_prices["VendorName"] = purchase_prices["VendorName"].str.strip()

In [46]:
vendor_check = purchases.groupby("VendorNumber")["VendorName"].nunique()
vendor_check[vendor_check >1]

Series([], Name: VendorName, dtype: int64)

### Vendor Name Consistency

I checked whether each `VendorNumber` was linked to a single vendor name. At first, a few vendor numbers appeared to have multiple names, which suggested there might be inconsistencies in the data.

After inspecting the values more closely, I found that some of the differences were caused by extra spaces at the end of the vendor name. I cleaned this by trimming whitespace from the `VendorName` field.

After this step, each vendor number was associated with a single consistent vendor name. Since VendorNumber serves as the primary identifier in the dataset, all vendor-level analysis will be based on this field to ensure consistent results.